# Notebook 01: Data Extraction and Exploratory Data Analysis

## Overview
This notebook initializes the research pipeline by loading sex-specific mortality data from the **Human Mortality Database (HMD)** for the 6-country frontier cluster: Switzerland (CHE), Sweden (SWE), Norway (NOR), West Germany (DEUTW), Netherlands (NLD), and Japan (JPN).

Unlike Project 04 (which used both-sexes-combined data), this project works with **Male and Female mortality separately**, enabling sex-specific longevity projections required for production-grade L&H applications (annuities, pension buy-outs, longevity swaps).

## Objectives
1. **Data Parsing**: Load raw HMD `.txt` files containing age-specific death rates ($m_{x,t}$) with Female, Male, and Total columns. Apply cleaning (age cap at 90, missing value handling, epsilon stabilization).
2. **Exploratory Analysis**: Generate comparative visualizations of log-mortality across the cluster, split by sex, to identify convergence patterns, structural breaks, and sex-specific dynamics.
3. **Data Validation**: Produce mortality surface heatmaps and cross-country comparisons to confirm data integrity and highlight the post-2011 deceleration.
4. **Export**: Save cleaned, standardized matrices for downstream modeling (Li-Lee baseline and AINN training).

## 1.1: Setup & Configuration

In [ ]:
# Reproducibility
import sys
sys.path.append('../src')
from reproducibility import set_seed
set_seed(42)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from style_config import set_style, save_dual, COUNTRIES

# Set notebook style
set_style("notebook")

# Paths
RAW_DATA_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"
FIGURES_DIR = "../reports/figures/"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Project parameters (consistent with Project 04)
YEAR_START = 1956
YEAR_END = 2020
AGE_START = 0
AGE_END = 90
EPSILON = 1e-10  # Numerical stability for log-transform

print(f"Cluster: {list(COUNTRIES.values())}")
print(f"Time window: {YEAR_START}-{YEAR_END}")
print(f"Age range: {AGE_START}-{AGE_END}")
print(f"Sex: Male, Female (separate)")

## 1.2: Data Parsing and Cleaning

In [ ]:
def load_hmd_mortality(country_code):
    """
    Parse HMD Mx_1x1.txt file for a given country.
    
    Returns a DataFrame with columns: Year, Age, Female, Male, Total.
    Filters to the project time window and age range.
    Handles missing values ('.') and the '110+' age group.
    """
    file_path = os.path.join(RAW_DATA_DIR, f"{country_code}_Mx_1x1.txt")
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing: {file_path}")
    
    # HMD standard: skip first 2 lines, whitespace separator, '.' as NaN
    df = pd.read_csv(file_path, sep='\\s+', skiprows=2, na_values='.')
    
    # Structural cleaning: Age '110+' to integer 110, then filter
    df['Age'] = df['Age'].astype(str).str.replace('+', '', regex=False).astype(int)
    
    # Type cleaning: ensure numeric
    for col in ['Female', 'Male', 'Total']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Filter to project window
    df = df[(df['Year'] >= YEAR_START) & (df['Year'] <= YEAR_END)]
    df = df[(df['Age'] >= AGE_START) & (df['Age'] <= AGE_END)]
    
    # Replace remaining NaN and zeros with epsilon (for log-stability)
    for col in ['Female', 'Male', 'Total']:
        df[col] = df[col].clip(lower=EPSILON)
    
    df = df.reset_index(drop=True)
    return df


# Load all countries
mortality_data = {}
for code in COUNTRIES:
    mortality_data[code] = load_hmd_mortality(code)
    n_years = mortality_data[code]["Year"].nunique()
    n_ages = mortality_data[code]["Age"].nunique()
    print(f"{code} ({COUNTRIES[code]}): {n_years} years x {n_ages} ages = {len(mortality_data[code]):,} rows")

print(f"\nAll {len(COUNTRIES)} countries loaded successfully.")

## 1.3: Data Validation

In [ ]:
# Verify consistent dimensions across the cluster
print("Dimension check (should be identical for all countries):")
print("-" * 50)
for code, df in mortality_data.items():
    years_list = sorted(df["Year"].unique())
    ages_list = sorted(df["Age"].unique())
    print(f"{code}: Years [{years_list[0]}-{years_list[-1]}], Ages [{ages_list[0]}-{ages_list[-1]}], Shape: {df.shape}")

# Check for any remaining NaN
print("\nMissing values check:")
for code, df in mortality_data.items():
    n_nan = df[["Female", "Male", "Total"]].isna().sum().sum()
    print(f"{code}: {n_nan} NaN values")

## 1.4: Log-Mortality Matrix Construction

We construct the mortality matrices in log-space: $\ln(m_{x,t})$. This linearizes the exponential Gompertz relationship and ensures positivity upon back-transformation.

In [ ]:
def build_log_mortality_matrix(df, sex="Male"):
    """
    Pivot mortality data into a matrix: rows = ages, columns = years.
    Returns log-transformed matrix.
    """
    pivot = df.pivot(index="Age", columns="Year", values=sex)
    return np.log(pivot.values)


# Build matrices for all countries and both sexes
log_matrices = {}
for code in COUNTRIES:
    log_matrices[code] = {
        "Male": build_log_mortality_matrix(mortality_data[code], "Male"),
        "Female": build_log_mortality_matrix(mortality_data[code], "Female"),
        "Total": build_log_mortality_matrix(mortality_data[code], "Total"),
    }

# Verify dimensions
ages = sorted(mortality_data["CHE"]["Age"].unique())
years = sorted(mortality_data["CHE"]["Year"].unique())
print(f"Matrix dimensions: {len(ages)} ages x {len(years)} years")
print(f"Ages: {ages[0]} to {ages[-1]}")
print(f"Years: {years[0]} to {years[-1]}")
print(f"\nExample (CHE Male): shape = {log_matrices['CHE']['Male'].shape}")
print(f"Range of ln(m_x): [{log_matrices['CHE']['Male'].min():.2f}, {log_matrices['CHE']['Male'].max():.2f}]")

## 1.5: Exploratory Visualization

### Log-mortality trends at age 65 (Male vs Female)
Comparative view of the cluster's mortality dynamics at a key actuarial age, split by sex.

In [ ]:
def plot_log_mortality_comparison(age_idx, age_val):
    """Plot log-mortality at a given age for all countries, Male vs Female."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    
    for sex_idx, sex in enumerate(["Male", "Female"]):
        ax = axes[sex_idx]
        for i, (code, name) in enumerate(COUNTRIES.items()):
            matrix = log_matrices[code][sex]
            ax.plot(years, matrix[age_idx, :], label=name, linewidth=1.5)
        
        ax.set_title(f"{sex} — Age {age_val}")
        ax.set_xlabel("Year")
        if sex_idx == 0:
            ax.set_ylabel("ln($m_x$)")
        ax.legend(loc="upper right", framealpha=0.9)
        ax.axvline(x=2011, color="grey", linestyle="--", alpha=0.5)
    
    fig.suptitle(f"Log-Mortality at Age {age_val}: Male vs Female (6-Country Cluster)", fontsize=14, y=1.02)
    plt.tight_layout()
    save_dual(fig, "fig01_log_mortality_age65_sex_comparison")
    plt.show()


plot_log_mortality_comparison(age_idx=65, age_val=65)

## 1.6: Mortality Surface (Switzerland)

Heatmap of the Swiss log-mortality surface to verify data integrity and visualize the age-period structure.

In [ ]:
def plot_mortality_surface(code, sex="Male"):
    """Plot log-mortality surface as a heatmap."""
    matrix = log_matrices[code][sex]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(
        matrix,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[years[0], years[-1], ages[0], ages[-1]]
    )
    
    ax.set_xlabel("Year")
    ax.set_ylabel("Age")
    ax.set_title(f"{COUNTRIES[code]} ({sex}) — Log-Mortality Surface")
    plt.colorbar(im, ax=ax, label="ln($m_x$)")
    
    plt.tight_layout()
    save_dual(fig, f"fig02_mortality_surface_{code}_{sex.lower()}")
    plt.show()


plot_mortality_surface("CHE", "Male")
plot_mortality_surface("CHE", "Female")

## 1.7: Sex Differential Analysis

The male-female mortality gap is a key actuarial quantity. We visualize how it evolves over time for the cluster.

In [ ]:
def plot_sex_differential(age_idx, age_val):
    """Plot the Male-Female log-mortality differential over time."""
    fig, ax = plt.subplots()
    
    for i, (code, name) in enumerate(COUNTRIES.items()):
        diff = log_matrices[code]["Male"][age_idx, :] - log_matrices[code]["Female"][age_idx, :]
        ax.plot(years, diff, label=name, linewidth=1.5)
    
    ax.axhline(y=0, color="grey", linestyle="-", alpha=0.3)
    ax.set_xlabel("Year")
    ax.set_ylabel("ln($m_x^M$) - ln($m_x^F$)")
    ax.set_title(f"Male-Female Mortality Gap at Age {age_val}")
    ax.legend(loc="upper right", framealpha=0.9)
    
    plt.tight_layout()
    save_dual(fig, f"fig03_sex_differential_age{age_val}")
    plt.show()


plot_sex_differential(age_idx=65, age_val=65)

## 1.8: Export Processed Data

Save cleaned mortality matrices for downstream notebooks (Li-Lee baseline, AINN training).

In [ ]:
# Save individual country DataFrames
for code, df in mortality_data.items():
    output_path = os.path.join(PROCESSED_DIR, f"{code}_mortality_clean.csv")
    df.to_csv(output_path, index=False)

# Save log-mortality matrices as .npy for fast loading
for code in COUNTRIES:
    for sex in ["Male", "Female", "Total"]:
        output_path = os.path.join(PROCESSED_DIR, f"{code}_log_mx_{sex.lower()}.npy")
        np.save(output_path, log_matrices[code][sex])

# Save metadata
import json as json_lib
metadata = {
    "countries": list(COUNTRIES.keys()),
    "years": [int(y) for y in years],
    "ages": [int(a) for a in ages],
    "year_start": YEAR_START,
    "year_end": YEAR_END,
    "age_start": AGE_START,
    "age_end": AGE_END,
    "epsilon": EPSILON,
}
with open(os.path.join(PROCESSED_DIR, "metadata.json"), "w") as f:
    json_lib.dump(metadata, f, indent=2)

print(f"Exported {len(COUNTRIES)} countries x 3 sexes = {len(COUNTRIES) * 3} matrices")
print(f"Saved to: {PROCESSED_DIR}")
print("\nNotebook 01 complete. Proceed to Notebook 02 (Actuarial Benchmarking).")